# Setup

In [ ]:
"""01_eda.ipynb — Causal exploratory data analysis of Hillstrom.

Goals:
1. Verify experimental properties (treatment balance, randomization, overlap).
2. Compute naive ATE as a sanity floor for all downstream estimators.
3. Look for heterogeneous treatment effects by eye, so we know what the
   causal models in Phase 2-3 ought to be learning.
"""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from uplift.data import load_raw
from uplift.treatment import (
    FEATURE_COLS,
    make_binary_treatment,
    make_xty,
    naive_ate,
)

# Project paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Plotting defaults — clean, readable, not too noisy
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"

# Seed for any random ops in this notebook
RNG = np.random.default_rng(42)

# Load and look

In [ ]:
df = load_raw()
print(f"shape: {df.shape}")
print(f"memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head()

# Descriptives

In [ ]:
df.describe(include="all").T

# Treatment balance check

In [ ]:
arm_counts = df["segment"].value_counts()
arm_proportions = df["segment"].value_counts(normalize=True)

balance_table = pd.DataFrame(
    {
        "count": arm_counts,
        "proportion": arm_proportions.round(4),
    }
)
print(balance_table)

# Visual
fig, ax = plt.subplots(figsize=(7, 3.5))
arm_proportions.plot.barh(ax=ax, color=["#888", "#4c78a8", "#e45756"])
ax.axvline(1 / 3, color="black", linestyle="--", linewidth=1, label="expected (1/3)")
ax.set_xlabel("Proportion of customers")
ax.set_ylabel("")
ax.set_title("Arm assignment proportions")
ax.legend()
fig.savefig(FIG_DIR / "01_arm_balance.png")
plt.show()

In [ ]:
T = make_binary_treatment(df)
print(f"binary treatment rate: {T.mean():.4f}")
print(f"  treated (any email): {(T == 1).sum():,}")
print(f"  control (no email):  {(T == 0).sum():,}")

# Outcome rates per arm (the naive ATE chart)

In [ ]:
arm_outcomes = (
    df.groupby("segment", observed=True)[["visit", "conversion", "spend"]]
    .mean()
    .reindex(["No E-Mail", "Mens E-Mail", "Womens E-Mail"])
)
print("Outcome means by arm:")
print(arm_outcomes.round(4))

# And the binarized version — this is the headline naive ATE
print("\nNaive ATE (any-email vs none):")
for outcome in ["visit", "conversion", "spend"]:
    result = naive_ate(df, outcome)
    lift = result["ate"]
    se = result["ate_se"]
    t_stat = lift / se
    print(
        f"  {outcome:10s}  treated={result['y_treated']:.4f}  "
        f"control={result['y_control']:.4f}  "
        f"ATE={lift:+.4f}  SE={se:.4f}  t={t_stat:.1f}"
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, outcome in zip(axes, ["visit", "conversion", "spend"]):
    arm_outcomes[outcome].plot.bar(
        ax=ax,
        color=["#888", "#4c78a8", "#e45756"],
        rot=15,
    )
    ax.set_title(f"Mean {outcome} by arm")
    ax.set_xlabel("")

fig.suptitle("Outcomes are higher in both treated arms — the experiment worked", y=1.02)
fig.savefig(FIG_DIR / "02_outcome_by_arm.png")
plt.show()

# Covariate balance (Love plot)

In [ ]:
def standardized_mean_diff(df: pd.DataFrame, T: pd.Series, col: str) -> float:
    x1 = df.loc[T == 1, col].astype(float)
    x0 = df.loc[T == 0, col].astype(float)
    # Pooled standard deviation
    s = np.sqrt((x1.var(ddof=1) + x0.var(ddof=1)) / 2)
    if s == 0:
        return 0.0
    return (x1.mean() - x0.mean()) / s


# Encode categorical features as dummies for the balance check
df_for_smd = pd.get_dummies(df[FEATURE_COLS], drop_first=False)
smds = pd.Series(
    {col: standardized_mean_diff(df_for_smd, T, col) for col in df_for_smd.columns}
).sort_values()

fig, ax = plt.subplots(figsize=(8, 0.3 * len(smds) + 1))
colors = ["#e45756" if abs(v) > 0.1 else "#4c78a8" for v in smds]
ax.hlines(y=range(len(smds)), xmin=0, xmax=smds.values, colors=colors)
ax.scatter(smds.values, range(len(smds)), color=colors, s=40)
ax.set_yticks(range(len(smds)))
ax.set_yticklabels(smds.index)
ax.axvline(0, color="black", linewidth=0.8)
ax.axvline(0.1, color="gray", linestyle="--", linewidth=0.8)
ax.axvline(-0.1, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("Standardized mean difference (treated − control)")
ax.set_title("Covariate balance: SMDs should sit between the dashed lines")
fig.savefig(FIG_DIR / "03_love_plot.png")
plt.show()

print(f"\nMax |SMD|: {smds.abs().max():.4f}")

# Randomization test via prediction

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score

X_dummies = pd.get_dummies(df[FEATURE_COLS], drop_first=True)
clf = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
auc_scores = cross_val_score(clf, X_dummies, T, cv=5, scoring="roc_auc")

print("5-fold CV AUC for predicting treatment from covariates:")
print(f"  mean: {auc_scores.mean():.4f}")
print(f"  std:  {auc_scores.std():.4f}")
print(f"  values: {auc_scores.round(4)}")
print(
    f"\nInterpretation: AUC ≈ 0.5 means treatment is unpredictable from "
    f"covariates,\nwhich is what we expect under randomization."
)

# Outcome rate by single covariates

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, col in zip(axes.flat, FEATURE_COLS):
    if df[col].dtype.name == "category" or df[col].nunique() < 15:
        # Categorical — bar chart
        rates = df.groupby(col, observed=True)["visit"].mean().sort_values()
        rates.plot.bar(ax=ax, color="#4c78a8")
        ax.set_title(f"Visit rate by {col}")
        ax.set_ylabel("P(visit)")
        ax.tick_params(axis="x", rotation=30)
    else:
        # Numeric — bin and plot
        binned = pd.cut(df[col], bins=10)
        rates = df.groupby(binned, observed=True)["visit"].mean()
        rates.plot(ax=ax, marker="o", color="#4c78a8")
        ax.set_title(f"Visit rate by {col} (binned)")
        ax.set_ylabel("P(visit)")
        ax.tick_params(axis="x", rotation=30)

fig.suptitle("Marginal visit rates by feature — where the predictive signal is", y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "04_visit_by_feature.png")
plt.show()

# The persuadable structure plot

In [ ]:
features_to_plot = ["recency", "history_segment", "channel", "zip_code"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for ax, col in zip(axes, features_to_plot):
    if df[col].dtype.name == "category" or df[col].nunique() < 15:
        grouped = df.groupby([col, T.rename("T")], observed=True)["visit"].mean().unstack("T")
        grouped.columns = ["control", "treated"]
        grouped = grouped.sort_values("control")
        x = np.arange(len(grouped))
        width = 0.35
        ax.bar(x - width / 2, grouped["control"], width, color="#888", label="control")
        ax.bar(x + width / 2, grouped["treated"], width, color="#4c78a8", label="treated")
        ax.set_xticks(x)
        ax.set_xticklabels(grouped.index, rotation=30, ha="right")
    else:
        binned = pd.cut(df[col], bins=8)
        grouped = df.groupby([binned, T.rename("T")], observed=True)["visit"].mean().unstack("T")
        grouped.columns = ["control", "treated"]
        grouped.index = [f"{int(i.left)}-{int(i.right)}" for i in grouped.index]
        ax.plot(range(len(grouped)), grouped["control"], "o-", color="#888", label="control")
        ax.plot(range(len(grouped)), grouped["treated"], "o-", color="#4c78a8", label="treated")
        ax.set_xticks(range(len(grouped)))
        ax.set_xticklabels(grouped.index, rotation=30)

    ax.set_title(f"Visit rate by {col} × treatment")
    ax.set_ylabel("P(visit)")
    ax.legend(loc="best", fontsize=9)

fig.suptitle(
    "Treatment effect heterogeneity: the gap between bars/lines is a local CATE",
    y=1.03,
)
fig.tight_layout()
fig.savefig(FIG_DIR / "05_heterogeneity.png")
plt.show()

# Quick summary cell

In [ ]:
print("=" * 60)
print("Phase 1 EDA Summary")
print("=" * 60)
print(f"Total customers:         {len(df):,}")
print(f"Treated (any email):     {(T == 1).sum():,} ({T.mean():.1%})")
print(f"Control:                 {(T == 0).sum():,} ({(1 - T).mean():.1%})")
print()
print(
    f"Naive ATE on visit:      +{naive_ate(df, 'visit')['ate']:.4f} (t = {naive_ate(df, 'visit')['ate'] / naive_ate(df, 'visit')['ate_se']:.1f})"
)
print(
    f"Naive ATE on spend:      ${naive_ate(df, 'spend')['ate']:+.4f} (t = {naive_ate(df, 'spend')['ate'] / naive_ate(df, 'spend')['ate_se']:.1f})"
)
print()
print(f"Max covariate SMD:       {smds.abs().max():.4f} (< 0.10 ⇒ balanced)")
print(f"AUC predicting T from X: {auc_scores.mean():.4f} (~0.50 ⇒ randomized)")
print()
print("Strong heterogeneity visible in: recency, channel, history_segment")
print("Weak heterogeneity in: zip_code, mens/womens flags")